# 10. Cierre final en test del ensamble

**Objetivo:** ejecutar inferencia `predict-only` del ensamble recongelado en `dev`, sin reentrenar ni ajustar pesos durante la evaluacion, y exportar resultados finales sobre el split seleccionado.

**Uso esperado:**

- Por defecto corre en `dev` para validación seca.
- Para abrir `test` se debe ejecutar explícitamente con:

```bash
FINAL_EVAL_SPLIT=test FINAL_PERMITIR_TEST=1 .venv/bin/jupyter nbconvert --to notebook --execute notebooks/pipeline/10_cierre_final_test_ensamble.ipynb --inplace
```

**Restricciones metodológicas:**

- No entrena modelos.
- No modifica reglas clínicas.
- No cambia pesos del ensamble durante la inferencia; usa los pesos recongelados en `dev`.
- No usa `test` para selección.
- El bootstrap agrupado por paciente, si se activa, solo estima incertidumbre; no recalibra el modelo.


## Estado de ejecución

La apertura final de `test` ya fue realizada una sola vez con el run:

- `data/outputs/cierre_final_test_ensamble_512_20260606_1640/`

Resultado agregado congelado: `n_eval=385`, `18` pacientes, Macro-F1 `0.555807`, balanced accuracy `0.553883` y weighted F1 `0.671349`. La matriz de confusión fue `[[29, 69], [54, 233]]` para el orden de clases `ansiedad`, `depresion`.

Este notebook permanece como fuente canónica para reproducir la inferencia final, pero no debe reejecutarse sobre `test` para reajustar pesos, modelos, filtros ni reglas. Para controles secos, usar `FINAL_EVAL_SPLIT=dev`.


## Contrato de cierre

Modelo cerrado en `dev`:

- Rama contextual: `ROBERTA_CLINICAL max_length=512`, peso `0.65`.
- Rama simbólica regionalizada: `py RF`, peso `0.15`.
- Rama simbólica core con late fusion LLM: `core RF`, peso `0.20`.

El notebook carga artefactos congelados y produce predicciones finales por weighted soft voting.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from packaging.version import Version
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
import sklearn

# Resolver raiz del repositorio de forma robusta.
def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'data').exists() and (p / 'notebooks').exists() and (p / 'scripts').exists():
            return p
    raise RuntimeError('No se pudo resolver la raiz del repositorio.')

REPO = find_repo_root(Path.cwd())
os.chdir(REPO)

sys.path.insert(0, str(REPO))
from utils_shared import setup_paths  # noqa: E402

paths = setup_paths()
DATA_PATH = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']
OUTPUTS_PATH = paths['OUTPUTS_PATH']

# Este notebook depende de modelos joblib guardados con scikit-learn 1.7.2.
if Version(sklearn.__version__) < Version('1.7.0'):
    raise RuntimeError(
        f'Entorno incompatible: scikit-learn={sklearn.__version__}. '
        'Ejecutar con .venv/bin/python o un kernel con scikit-learn>=1.7.'
    )

LABELS = ['ansiedad', 'depresion']
PROB_COLS = [f'prob_{c}' for c in LABELS]
MAX_LENGTH = 512
HIDDEN_SIZE_ROBERTA = 768
PROB_REPRO_TOL = float(os.getenv('FINAL_ROBERTA_REPRO_TOL', '1e-5'))
REQUIRE_DEV_ROBERTA_REPRO = os.getenv('FINAL_REQUIRE_DEV_ROBERTA_REPRO', '1') != '0'

EVAL_SPLIT = os.getenv('FINAL_EVAL_SPLIT', 'dev').strip().lower()
if EVAL_SPLIT not in {'dev', 'test'}:
    raise ValueError('FINAL_EVAL_SPLIT debe ser dev o test.')
if EVAL_SPLIT == 'test' and os.getenv('FINAL_PERMITIR_TEST', '0') != '1':
    raise RuntimeError('TEST protegido: para abrirlo definir FINAL_EVAL_SPLIT=test y FINAL_PERMITIR_TEST=1.')
if EVAL_SPLIT == 'test' and os.getenv('FINAL_DEV_REPRO_OK', '0') != '1':
    raise RuntimeError(
        'TEST protegido: antes de abrir test debe existir una corrida dev que reproduzca la rama '
        'ROBERTA_CLINICAL oficial. Ejecutar primero FINAL_EVAL_SPLIT=dev y documentar '
        'el check; solo luego usar FINAL_DEV_REPRO_OK=1.'
    )

RUN_ID_ENV = os.getenv('CIERRE_FINAL_RUN_ID', '').strip()
if RUN_ID_ENV:
    RUN_ID = RUN_ID_ENV
else:
    prefix = 'cierre_final_test_xai' if EVAL_SPLIT == 'test' else 'cierre_final_pretest_dev'
    RUN_ID = pd.Timestamp.now().strftime(f'{prefix}_%Y%m%d_%H%M%S')

OUT_DIR = OUTPUTS_PATH / RUN_ID
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

BOOTSTRAP_N = int(os.getenv('FINAL_BOOTSTRAP_N', '1000' if EVAL_SPLIT == 'test' else '0'))
BOOTSTRAP_SEED = int(os.getenv('FINAL_BOOTSTRAP_SEED', '42'))
BATCH_SIZE = int(os.getenv('FINAL_BATCH_SIZE', '8'))

print('REPO:', REPO)
print('Python:', sys.executable)
print('scikit-learn:', sklearn.__version__)
print('EVAL_SPLIT:', EVAL_SPLIT)
print('RUN_ID:', RUN_ID)
print('OUT_DIR:', OUT_DIR)
print('BOOTSTRAP_N:', BOOTSTRAP_N)
print('REQUIRE_DEV_ROBERTA_REPRO:', REQUIRE_DEV_ROBERTA_REPRO)
print('PROB_REPRO_TOL:', PROB_REPRO_TOL)


In [ ]:
# Artefactos congelados del ensamble dev.
ROBERTA_CHECKPOINT = Path(os.getenv(
    'ROBERTA_CHECKPOINT_PATH',
    str(DATA_PATH / 'checkpoints' / 'roberta_clinical' / 'checkpoint-210'),
))

ARTIFACTS = {
    'cierre_dev_dir': OUTPUTS_PATH / 'cierre_dev_recongelado_roberta_512_20260606_160946',
    'roberta_checkpoint': ROBERTA_CHECKPOINT,
    'roberta_dev_reference_pred': DATA_PATH / 'roberta_clinical_predicciones_dev.csv',
    'simbolico_py_model': OUTPUTS_PATH / 'train_A_llm0_sent0_beto0_tpl0_20260401_112720' / 'modelo_py_RF.joblib',
    'simbolico_py_cols': OUTPUTS_PATH / 'train_A_llm0_sent0_beto0_tpl0_20260401_112720' / 'py_X_cols.json',
    'simbolico_py_features': DATA_PATH / 'processed' / 'fe_20260401_093631_py' / 'features_py.parquet',
    'simbolico_core_model': OUTPUTS_PATH / 'train_A_llm1_sent0_beto0_tpl0_20260401_112720' / 'modelo_core_RF.joblib',
    'simbolico_core_cols': OUTPUTS_PATH / 'train_A_llm1_sent0_beto0_tpl0_20260401_112720' / 'core_X_cols.json',
    'simbolico_core_features': DATA_PATH / 'processed' / 'fe_20260401_093631_core' / 'features_core.parquet',
}

missing = [str(p) for p in ARTIFACTS.values() if not Path(p).exists()]
if missing:
    raise FileNotFoundError('Faltan artefactos congelados:\n' + '\n'.join(missing))

BRANCH_SPECS = [
    {
        'key': 'ctx_roberta_clinical_512',
        'nombre': 'ROBERTA_CLINICAL max_length=512',
        'tipo': 'contextual',
        'peso': 0.65,
        'max_length': MAX_LENGTH,
        'hidden_size': HIDDEN_SIZE_ROBERTA,
    },
    {
        'key': 'simbolico_py_rf',
        'nombre': 'Simbólico regionalizado py RF balanceado',
        'tipo': 'simbolica_regionalizada',
        'peso': 0.15,
        'model_path': ARTIFACTS['simbolico_py_model'],
        'cols_path': ARTIFACTS['simbolico_py_cols'],
        'features_path': ARTIFACTS['simbolico_py_features'],
        'profile': 'py',
    },
    {
        'key': 'simbolico_core_rf_late_fusion_llm',
        'nombre': 'Simbólico core RF con late fusion LLM',
        'tipo': 'simbolica_core_late_fusion_llm',
        'peso': 0.20,
        'model_path': ARTIFACTS['simbolico_core_model'],
        'cols_path': ARTIFACTS['simbolico_core_cols'],
        'features_path': ARTIFACTS['simbolico_core_features'],
        'profile': 'core',
    },
]

peso_total = sum(float(s['peso']) for s in BRANCH_SPECS)
if not np.isclose(peso_total, 1.0):
    raise ValueError(f'Los pesos del ensamble no suman 1.0: {peso_total}')

for k, p in ARTIFACTS.items():
    print(k, '->', p)

print('Nota metodologica: roberta_checkpoint debe reproducir data/roberta_clinical_predicciones_dev.csv en dry-run dev antes de abrir test.')


In [ ]:
# Carga del split evaluado.
split_path = SPLITS_PATH / f'{EVAL_SPLIT}_denoised.csv'
if not split_path.exists():
    raise FileNotFoundError(split_path)

base_df = pd.read_csv(split_path)
required_base = {'row_id', 'patient_id', 'etiqueta', 'texto'}
missing_base = required_base - set(base_df.columns)
if missing_base:
    raise ValueError(f'Faltan columnas en {split_path}: {missing_base}')

base_df = base_df.copy()
base_df['row_id'] = base_df['row_id'].astype(int)
base_df['etiqueta'] = base_df['etiqueta'].astype(str)

if not set(base_df['etiqueta']).issubset(set(LABELS)):
    raise ValueError(f'Etiquetas inesperadas: {sorted(set(base_df["etiqueta"]) - set(LABELS))}')
if base_df['row_id'].duplicated().any():
    raise ValueError('row_id duplicados en split evaluado.')

print('Split:', EVAL_SPLIT)
print('n_eval:', len(base_df))
print('pacientes:', base_df['patient_id'].nunique())
print('clases:', base_df['etiqueta'].value_counts().to_dict())
display(base_df[['row_id', 'patient_id', 'etiqueta']].head())


In [ ]:
def metricas_globales(y_true, y_pred) -> dict:
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    }


def metricas_por_clase(nombre: str, y_true, y_pred) -> pd.DataFrame:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    return pd.DataFrame({
        'modelo': nombre,
        'split': EVAL_SPLIT,
        'clase': LABELS,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'support': support,
    })


def pred_from_probs(prob_matrix: np.ndarray) -> list[str]:
    idx = np.argmax(prob_matrix, axis=1)
    return [LABELS[int(i)] for i in idx]


def validar_predicciones(nombre: str, pred_df: pd.DataFrame, base: pd.DataFrame) -> dict:
    required = {'row_id', 'y_true', 'y_pred', *PROB_COLS}
    missing = required - set(pred_df.columns)
    if missing:
        raise ValueError(f'{nombre}: faltan columnas {missing}')

    pred = pred_df.copy()
    pred['row_id'] = pred['row_id'].astype(int)
    pred = pred.sort_values('row_id').reset_index(drop=True)
    base_sorted = base[['row_id', 'etiqueta']].sort_values('row_id').reset_index(drop=True)

    same_n = len(pred) == len(base_sorted)
    same_row_id = same_n and pred['row_id'].equals(base_sorted['row_id'])
    same_y_true = same_n and pred['y_true'].astype(str).equals(base_sorted['etiqueta'].astype(str))
    valid_preds = set(pred['y_pred'].astype(str)).issubset(set(LABELS))
    prob_sums = pred[PROB_COLS].sum(axis=1)
    prob_sum_max_abs_error = float(np.max(np.abs(prob_sums - 1.0))) if len(prob_sums) else np.nan
    prob_ok = bool(np.allclose(prob_sums, 1.0, atol=1e-4))

    checks = {
        'rama': nombre,
        'n_pred': int(len(pred)),
        'n_base': int(len(base_sorted)),
        'same_n': bool(same_n),
        'same_row_id': bool(same_row_id),
        'same_y_true': bool(same_y_true),
        'valid_preds': bool(valid_preds),
        'prob_ok': bool(prob_ok),
        'prob_sum_max_abs_error': prob_sum_max_abs_error,
    }
    if not all([same_n, same_row_id, same_y_true, valid_preds, prob_ok]):
        raise ValueError(f'Fallo alineacion {nombre}: {checks}')
    return checks


def git_out(args):
    try:
        return subprocess.check_output(args, cwd=REPO, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return ''


def rel(path):
    try:
        return str(Path(path).resolve().relative_to(REPO.resolve()))
    except Exception:
        return str(path)


In [ ]:
# Rama contextual: inferencia desde checkpoint congelado.
def resolve_device():
    import torch
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def predict_roberta(base: pd.DataFrame) -> pd.DataFrame:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    checkpoint = ARTIFACTS['roberta_checkpoint']
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
    model.eval()
    device = resolve_device()
    model.to(device)

    id2label = {int(k): str(v) for k, v in model.config.id2label.items()}
    if [id2label[i] for i in sorted(id2label)] != LABELS:
        raise ValueError(f'Orden de clases inesperado en checkpoint: {id2label}')

    texts = base['texto'].astype(str).tolist()
    all_probs = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[start:start + BATCH_SIZE]
        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.nn.functional.softmax(logits, dim=-1).detach().cpu().numpy()
        all_probs.append(probs)

    probs = np.vstack(all_probs)
    out = pd.DataFrame({
        'row_id': base['row_id'].astype(int).values,
        'y_true': base['etiqueta'].astype(str).values,
        'y_pred': pred_from_probs(probs),
        'prob_ansiedad': probs[:, 0],
        'prob_depresion': probs[:, 1],
    })
    return out



def comparar_roberta_con_referencia_dev(pred_df: pd.DataFrame) -> dict | None:
    """Valida que el checkpoint contextual reproduzca la prediccion oficial dev.

    Este check es una compuerta metodologica: no se debe abrir test si el checkpoint
    local produce otra rama contextual que la usada para cerrar el ensamble en dev.
    """
    if EVAL_SPLIT == 'test':
        cierre_manifest_path = ARTIFACTS['cierre_dev_dir'] / 'manifest.json'
        cierre_manifest = json.loads(cierre_manifest_path.read_text(encoding='utf-8'))
        summary = dict(cierre_manifest.get('check_reproduccion_roberta_dev') or {})
        summary['fuente_persistida'] = rel(cierre_manifest_path)
        if not summary.get('reproduce_reference'):
            raise RuntimeError('El cierre dev no acredita la reproduccion de la rama ROBERTA.')
        return summary

    ref_path = ARTIFACTS['roberta_dev_reference_pred']
    ref = pd.read_csv(ref_path)
    required = {'row_id', 'y_true', 'y_pred', *PROB_COLS}
    missing = required - set(ref.columns)
    if missing:
        raise ValueError(f'Referencia ROBERTA dev incompleta: {missing}')

    pred = pred_df[list(required)].copy().sort_values('row_id').reset_index(drop=True)
    ref = ref[list(required)].copy().sort_values('row_id').reset_index(drop=True)
    same_n = len(pred) == len(ref)
    same_row_id = same_n and pred['row_id'].astype(int).equals(ref['row_id'].astype(int))
    same_y_true = same_n and pred['y_true'].astype(str).equals(ref['y_true'].astype(str))
    y_pred_diff = int((pred['y_pred'].astype(str) != ref['y_pred'].astype(str)).sum()) if same_n else None
    prob_max_abs_diff = float((pred[PROB_COLS] - ref[PROB_COLS]).abs().to_numpy().max()) if same_n else None
    prob_mean_abs_diff = float((pred[PROB_COLS] - ref[PROB_COLS]).abs().to_numpy().mean()) if same_n else None
    prob_close = bool(prob_max_abs_diff is not None and prob_max_abs_diff <= PROB_REPRO_TOL)
    reproduce_reference = bool(same_n and same_row_id and same_y_true and y_pred_diff == 0 and prob_close)

    summary = {
        'split': EVAL_SPLIT,
        'checkpoint_path': rel(ARTIFACTS['roberta_checkpoint']),
        'reference_path': rel(ref_path),
        'same_n': bool(same_n),
        'same_row_id': bool(same_row_id),
        'same_y_true': bool(same_y_true),
        'y_pred_diff': y_pred_diff,
        'prob_max_abs_diff': prob_max_abs_diff,
        'prob_mean_abs_diff': prob_mean_abs_diff,
        'prob_tolerance': PROB_REPRO_TOL,
        'reproduce_reference': reproduce_reference,
    }

    check_path = OUT_DIR / 'check_reproduccion_roberta_dev.csv'
    pd.DataFrame([summary]).to_csv(check_path, index=False)

    if not reproduce_reference and same_n and same_row_id:
        diff = pred.copy()
        diff = diff.rename(columns={
            'y_pred': 'y_pred_checkpoint',
            'prob_ansiedad': 'prob_ansiedad_checkpoint',
            'prob_depresion': 'prob_depresion_checkpoint',
        })
        diff['y_pred_referencia'] = ref['y_pred'].astype(str).values
        diff['prob_ansiedad_referencia'] = ref['prob_ansiedad'].values
        diff['prob_depresion_referencia'] = ref['prob_depresion'].values
        diff['diff_prob_ansiedad_abs'] = (diff['prob_ansiedad_checkpoint'] - diff['prob_ansiedad_referencia']).abs()
        diff['diff_prob_depresion_abs'] = (diff['prob_depresion_checkpoint'] - diff['prob_depresion_referencia']).abs()
        diff = diff[
            (diff['y_pred_checkpoint'] != diff['y_pred_referencia'])
            | (diff['diff_prob_ansiedad_abs'] > PROB_REPRO_TOL)
            | (diff['diff_prob_depresion_abs'] > PROB_REPRO_TOL)
        ]
        diff_path = OUT_DIR / 'diferencias_roberta_dev_vs_referencia.csv'
        diff.to_csv(diff_path, index=False)
        summary['diff_path'] = rel(diff_path)

    print('Check reproduccion ROBERTA dev:', summary)
    if not reproduce_reference and REQUIRE_DEV_ROBERTA_REPRO:
        raise RuntimeError(
            'Bloqueo metodologico: el checkpoint ROBERTA_CLINICAL indicado no reproduce las '\
            'predicciones oficiales dev usadas en el cierre del ensamble 512. No abrir test '\
            'hasta localizar el checkpoint exacto o re-congelar formalmente una rama contextual reproducible.'
        )
    return summary

pred_roberta = predict_roberta(base_df)
validar_predicciones('ctx_roberta_clinical_512', pred_roberta, base_df)
check_repro_roberta = comparar_roberta_con_referencia_dev(pred_roberta)
roberta_path = OUT_DIR / f'predicciones_roberta_clinical_{EVAL_SPLIT}.csv'
pred_roberta.to_csv(roberta_path, index=False)
print('Exportado:', roberta_path)
display(pred_roberta.head())


In [ ]:
# Ramas simbolicas: inferencia desde modelos joblib congelados.
def predict_symbolic_branch(spec: dict, base: pd.DataFrame) -> pd.DataFrame:
    model = joblib.load(spec['model_path'])
    with open(spec['cols_path'], 'r', encoding='utf-8') as f:
        cols = json.load(f)

    features = pd.read_parquet(spec['features_path'])
    features = features.copy()
    features['row_id'] = features['row_id'].astype(int)

    # Mantener exactamente el orden del split evaluado.
    work = base[['row_id', 'etiqueta']].merge(features, on='row_id', how='left', validate='one_to_one')
    if len(work) != len(base):
        raise ValueError(f'{spec["key"]}: la union con features cambio el numero de filas.')
    if work[cols].isna().all(axis=1).any():
        raise ValueError(f'{spec["key"]}: hay filas sin features despues del merge.')

    missing_cols = [c for c in cols if c not in work.columns]
    if missing_cols:
        raise ValueError(f'{spec["key"]}: faltan columnas requeridas: {missing_cols[:20]}')

    X = work[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    probs_raw = model.predict_proba(X)
    class_to_pos = {str(cls): i for i, cls in enumerate(model.classes_)}
    probs = np.column_stack([probs_raw[:, class_to_pos[label]] for label in LABELS])

    out = pd.DataFrame({
        'row_id': base['row_id'].astype(int).values,
        'y_true': base['etiqueta'].astype(str).values,
        'y_pred': pred_from_probs(probs),
        'prob_ansiedad': probs[:, 0],
        'prob_depresion': probs[:, 1],
    })
    return out

pred_symbolic = {}
symbolic_paths = {}
for spec in BRANCH_SPECS:
    if spec['tipo'] == 'contextual':
        continue
    df_pred = predict_symbolic_branch(spec, base_df)
    validar_predicciones(spec['key'], df_pred, base_df)
    out_path = OUT_DIR / f'predicciones_{spec["key"]}_{EVAL_SPLIT}.csv'
    df_pred.to_csv(out_path, index=False)
    pred_symbolic[spec['key']] = df_pred
    symbolic_paths[f'predicciones_{spec["key"]}'] = out_path
    print('Exportado:', out_path)
    display(df_pred.head())


In [ ]:
# Ensamble weighted soft con validacion estricta.
branch_predictions = {'ctx_roberta_clinical_512': pred_roberta, **pred_symbolic}
check_rows = []
for spec in BRANCH_SPECS:
    check = validar_predicciones(spec['key'], branch_predictions[spec['key']], base_df)
    check['peso'] = spec['peso']
    check['tipo'] = spec['tipo']
    check_rows.append(check)

check_df = pd.DataFrame(check_rows)
check_path = OUT_DIR / f'check_alineacion_ensamble_{EVAL_SPLIT}.csv'
check_df.to_csv(check_path, index=False)
display(check_df)

base_sorted = base_df[['row_id', 'patient_id', 'etiqueta']].sort_values('row_id').reset_index(drop=True)
probs_final = np.zeros((len(base_sorted), len(LABELS)), dtype=float)
ens = base_sorted.rename(columns={'etiqueta': 'y_true'}).copy()

for spec in BRANCH_SPECS:
    key = spec['key']
    weight = float(spec['peso'])
    df_branch = branch_predictions[key].sort_values('row_id').reset_index(drop=True)
    branch_probs = df_branch[PROB_COLS].to_numpy(dtype=float)
    probs_final += weight * branch_probs
    ens[f'y_pred__{key}'] = df_branch['y_pred'].astype(str).values
    for j, label in enumerate(LABELS):
        ens[f'prob_{label}__{key}'] = branch_probs[:, j]
        ens[f'contrib_{label}__{key}'] = weight * branch_probs[:, j]

ens['prob_ansiedad'] = probs_final[:, 0]
ens['prob_depresion'] = probs_final[:, 1]
ens['y_pred'] = pred_from_probs(probs_final)
ens['error'] = ens['y_true'].astype(str) != ens['y_pred'].astype(str)

pred_ens_path = OUT_DIR / f'predicciones_ensamble_{EVAL_SPLIT}.csv'
ens.to_csv(pred_ens_path, index=False)
print('Exportado:', pred_ens_path)
display(ens.head())


In [ ]:
# Metricas finales del split evaluado.
y_true = ens['y_true'].astype(str)
y_pred = ens['y_pred'].astype(str)
metrics = metricas_globales(y_true, y_pred)
metrics_row = {
    'modelo': 'ensamble_weighted_soft_roberta_clinical_512_simbolico_py_rf_simbolico_core_rf_late_fusion_llm',
    'split': EVAL_SPLIT,
    'n_eval': int(len(ens)),
    'n_pacientes': int(ens['patient_id'].nunique()),
    **metrics,
}
metrics_df = pd.DataFrame([metrics_row])
metrics_path = OUT_DIR / f'metricas_{EVAL_SPLIT}.csv'
metrics_df.to_csv(metrics_path, index=False)

def per_class_for_model(name, df):
    return metricas_por_clase(name, df['y_true'].astype(str), df['y_pred'].astype(str))

per_class_rows = [per_class_for_model('ensamble_weighted_soft', ens)]
per_class_rows.append(per_class_for_model('ctx_roberta_clinical_512', pred_roberta))
for key, df_pred in pred_symbolic.items():
    per_class_rows.append(per_class_for_model(key, df_pred))
per_class_df = pd.concat(per_class_rows, ignore_index=True)
per_class_path = OUT_DIR / f'metricas_por_clase_{EVAL_SPLIT}.csv'
per_class_df.to_csv(per_class_path, index=False)

cm = confusion_matrix(y_true, y_pred, labels=LABELS)
cm_df = pd.DataFrame(cm, index=[f'true_{c}' for c in LABELS], columns=[f'pred_{c}' for c in LABELS])
cm_path = OUT_DIR / f'matriz_confusion_{EVAL_SPLIT}.csv'
cm_json_path = OUT_DIR / f'matriz_confusion_{EVAL_SPLIT}.json'
cm_md_path = OUT_DIR / f'matriz_confusion_{EVAL_SPLIT}.md'
cm_df.to_csv(cm_path)
cm_json_path.write_text(json.dumps({'labels': LABELS, 'matrix': cm.tolist()}, ensure_ascii=False, indent=2), encoding='utf-8')
cm_md_path.write_text('# Matriz de confusion\n\n' + cm_df.to_markdown() + '\n', encoding='utf-8')

print('Metricas globales')
display(metrics_df)
print('Metricas por clase')
display(per_class_df)
print('Matriz de confusion')
display(cm_df)


In [ ]:
# Incertidumbre por bootstrap agrupado por paciente.
# No reentrena ni recalibra: solo remuestrea pacientes ya evaluados.
bootstrap_paths = {}
if BOOTSTRAP_N > 0:
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    patients = ens['patient_id'].astype(str).unique()
    rows = []
    for b in range(BOOTSTRAP_N):
        sampled = rng.choice(patients, size=len(patients), replace=True)
        sampled_frames = []
        for pid in sampled:
            sampled_frames.append(ens[ens['patient_id'].astype(str) == str(pid)])
        boot = pd.concat(sampled_frames, ignore_index=True)
        m = metricas_globales(boot['y_true'].astype(str), boot['y_pred'].astype(str))
        rows.append({'bootstrap_iter': b, 'n_pacientes_remuestreados': len(sampled), 'n_notas': len(boot), **m})
    boot_df = pd.DataFrame(rows)
    summary_rows = []
    for metric in ['accuracy', 'macro_f1', 'weighted_f1', 'balanced_accuracy']:
        vals = boot_df[metric].dropna().to_numpy()
        summary_rows.append({
            'metric': metric,
            'mean': float(np.mean(vals)),
            'std': float(np.std(vals, ddof=1)),
            'ci95_low': float(np.quantile(vals, 0.025)),
            'ci95_high': float(np.quantile(vals, 0.975)),
            'n_bootstrap': int(len(vals)),
            'agrupado_por': 'patient_id',
        })
    boot_summary = pd.DataFrame(summary_rows)
    boot_path = OUT_DIR / f'bootstrap_metricas_{EVAL_SPLIT}.csv'
    boot_summary_path = OUT_DIR / f'bootstrap_resumen_{EVAL_SPLIT}.csv'
    boot_df.to_csv(boot_path, index=False)
    boot_summary.to_csv(boot_summary_path, index=False)
    bootstrap_paths = {'bootstrap_metricas': boot_path, 'bootstrap_resumen': boot_summary_path}
    display(boot_summary)
else:
    print('Bootstrap no ejecutado. Definir FINAL_BOOTSTRAP_N>0 para activarlo.')


## Auditoría descriptiva del resultado final

A partir de las predicciones ya congeladas, esta sección compara las ramas, caracteriza los errores, resume su concentración por paciente y selecciona casos técnicos A/B/C sin incluir texto clínico libre. También contrasta `dev` y `test` usando el manifiesto del recongelado. Estas operaciones son descriptivas: no reentrenan, no recalibran y no modifican la decisión final.


In [ ]:
# Auditoria descriptiva posterior a la prediccion final.
# Solo transforma predicciones ya congeladas; no reentrena ni reajusta el ensamble.
def resumen_modelo(nombre: str, pred: pd.DataFrame) -> dict:
    yt = pred['y_true'].astype(str)
    yp = pred['y_pred'].astype(str)
    precision, recall, f1, support = precision_recall_fscore_support(
        yt, yp, labels=LABELS, zero_division=0
    )
    matrix = confusion_matrix(yt, yp, labels=LABELS)
    return {
        'modelo': nombre,
        'n_eval': int(len(pred)),
        'n_pacientes': int(base_df['patient_id'].nunique()),
        **metricas_globales(yt, yp),
        'precision_ansiedad': float(precision[0]),
        'recall_ansiedad': float(recall[0]),
        'f1_ansiedad': float(f1[0]),
        'support_ansiedad': int(support[0]),
        'precision_depresion': float(precision[1]),
        'recall_depresion': float(recall[1]),
        'f1_depresion': float(f1[1]),
        'support_depresion': int(support[1]),
        'true_ansiedad_pred_ansiedad': int(matrix[0, 0]),
        'true_ansiedad_pred_depresion': int(matrix[0, 1]),
        'true_depresion_pred_ansiedad': int(matrix[1, 0]),
        'true_depresion_pred_depresion': int(matrix[1, 1]),
    }

branch_metric_rows = [resumen_modelo('ensamble_weighted_soft', ens)]
branch_metric_rows.append(resumen_modelo('ctx_roberta_clinical_512', pred_roberta))
for key, pred in pred_symbolic.items():
    branch_metric_rows.append(resumen_modelo(key, pred))
branch_metrics_df = pd.DataFrame(branch_metric_rows)
branch_metrics_path = OUT_DIR / f'metricas_globales_por_rama_{EVAL_SPLIT}.csv'
branch_metrics_df.to_csv(branch_metrics_path, index=False)

comparison_df = ens.copy()
comparison_df['margen_abs'] = (comparison_df['prob_ansiedad'] - comparison_df['prob_depresion']).abs()
comparison_df['ensamble_correcto'] = comparison_df['y_true'] == comparison_df['y_pred']
comparison_df['tipo_error'] = np.select(
    [
        comparison_df['ensamble_correcto'],
        (comparison_df['y_true'] == 'ansiedad') & (comparison_df['y_pred'] == 'depresion'),
    ],
    ['acierto', 'FN_ansiedad_pred_depresion'],
    default='FP_ansiedad_pred_depresion_real',
)
branch_pred_cols = [f'y_pred__{spec["key"]}' for spec in BRANCH_SPECS]
comparison_df['n_ramas_ansiedad'] = (comparison_df[branch_pred_cols] == 'ansiedad').sum(axis=1)
comparison_df['n_ramas_correctas'] = comparison_df[branch_pred_cols].eq(comparison_df['y_true'], axis=0).sum(axis=1)
comparison_df['desacuerdo_ramas'] = comparison_df[branch_pred_cols].nunique(axis=1) > 1
comparison_df['todas_ramas_erradas'] = comparison_df['n_ramas_correctas'] == 0
comparison_df['patron_ramas'] = (
    'ctx=' + comparison_df['y_pred__ctx_roberta_clinical_512'].astype(str)
    + '|py=' + comparison_df['y_pred__simbolico_py_rf'].astype(str)
    + '|core=' + comparison_df['y_pred__simbolico_core_rf_late_fusion_llm'].astype(str)
)
comparison_cols = [
    'row_id', 'patient_id', 'y_true', 'y_pred', 'prob_ansiedad', 'prob_depresion',
    'margen_abs', 'ensamble_correcto', 'error', 'tipo_error', 'n_ramas_ansiedad',
    'n_ramas_correctas', 'desacuerdo_ramas', 'todas_ramas_erradas', 'patron_ramas',
]
for spec in BRANCH_SPECS:
    comparison_cols.extend([
        f'y_pred__{spec["key"]}', f'prob_ansiedad__{spec["key"]}',
        f'prob_depresion__{spec["key"]}',
    ])
comparison_df = comparison_df[comparison_cols]
comparison_path = OUT_DIR / f'predicciones_comparadas_ensamble_{EVAL_SPLIT}.csv'
comparison_df.to_csv(comparison_path, index=False)
errors_df = comparison_df[comparison_df['error']].sort_values('margen_abs').copy()
errors_path = OUT_DIR / f'errores_ensamble_{EVAL_SPLIT}.csv'
errors_df.to_csv(errors_path, index=False)

patient_rows = []
for patient_id, group in comparison_df.groupby('patient_id', sort=False):
    patient_rows.append({
        'patient_id': patient_id,
        'n_notas': int(len(group)),
        'n_ansiedad_real': int((group['y_true'] == 'ansiedad').sum()),
        'n_depresion_real': int((group['y_true'] == 'depresion').sum()),
        'n_pred_ansiedad': int((group['y_pred'] == 'ansiedad').sum()),
        'n_pred_depresion': int((group['y_pred'] == 'depresion').sum()),
        'n_errores': int(group['error'].sum()),
        'error_rate': float(group['error'].mean()),
        'n_FN_ansiedad': int((group['tipo_error'] == 'FN_ansiedad_pred_depresion').sum()),
        'n_FP_ansiedad': int((group['tipo_error'] == 'FP_ansiedad_pred_depresion_real').sum()),
    })
patient_summary = pd.DataFrame(patient_rows).sort_values(['n_errores', 'error_rate'], ascending=False)
patient_summary_path = OUT_DIR / f'resumen_por_paciente_{EVAL_SPLIT}.csv'
patient_summary.to_csv(patient_summary_path, index=False)

case_specs = [
    ('A', 'A: ensamble corrige a ROBERTA/contextual; se elige el caso de menor margen final.', comparison_df['ensamble_correcto'] & (comparison_df['y_pred__ctx_roberta_clinical_512'] != comparison_df['y_true']), True),
    ('B', 'B: error de frontera; ensamble falla con el menor margen absoluto de probabilidad.', comparison_df['error'], True),
    ('C', 'C: error compartido por las tres ramas; se elige un caso de margen alto.', comparison_df['error'] & comparison_df['todas_ramas_erradas'], False),
]
case_rows = []
for case_name, criterion, mask, ascending in case_specs:
    candidates = comparison_df.loc[mask].sort_values('margen_abs', ascending=ascending)
    if candidates.empty:
        continue
    row = candidates.iloc[0]
    case_rows.append({
        'caso': case_name, 'criterio': criterion, 'row_id': row['row_id'],
        'patient_id': row['patient_id'], 'y_true': row['y_true'],
        'y_pred_ensamble': row['y_pred'], 'prob_ansiedad_ensamble': row['prob_ansiedad'],
        'prob_depresion_ensamble': row['prob_depresion'], 'margen_abs': row['margen_abs'],
        'y_pred_ctx_roberta': row['y_pred__ctx_roberta_clinical_512'],
        'prob_ansiedad_ctx_roberta': row['prob_ansiedad__ctx_roberta_clinical_512'],
        'y_pred_simbolico_py_rf': row['y_pred__simbolico_py_rf'],
        'prob_ansiedad_simbolico_py_rf': row['prob_ansiedad__simbolico_py_rf'],
        'y_pred_simbolico_core_rf_late_fusion_llm': row['y_pred__simbolico_core_rf_late_fusion_llm'],
        'prob_ansiedad_simbolico_core_rf_late_fusion_llm': row['prob_ansiedad__simbolico_core_rf_late_fusion_llm'],
        'tipo_error': row['tipo_error'], 'patron_ramas': row['patron_ramas'],
    })
cases_abc_df = pd.DataFrame(case_rows)
cases_abc_path = OUT_DIR / f'casos_abc_ensamble_{EVAL_SPLIT}.csv'
cases_abc_df.to_csv(cases_abc_path, index=False)

dev_test_path = None
if EVAL_SPLIT == 'test':
    dev_manifest = json.loads((ARTIFACTS['cierre_dev_dir'] / 'manifest.json').read_text(encoding='utf-8'))
    dev_metrics = dev_manifest['metricas_principales']
    test_metrics = branch_metric_rows[0]
    metrics_compare = [
        'accuracy', 'macro_f1', 'balanced_accuracy', 'weighted_f1',
        'f1_ansiedad', 'f1_depresion', 'recall_ansiedad', 'recall_depresion',
    ]
    dev_test_df = pd.DataFrame([
        {'metric': key, 'dev': dev_metrics[key], 'test': test_metrics[key],
         'delta_test_minus_dev': test_metrics[key] - dev_metrics[key]}
        for key in metrics_compare
    ])
    dev_test_path = OUT_DIR / 'comparacion_dev_test_ensamble.csv'
    dev_test_df.to_csv(dev_test_path, index=False)

error_report_path = OUT_DIR / f'reporte_analisis_errores_{EVAL_SPLIT}.md'
error_report_lines = [
    f'# Analisis de errores - cierre {EVAL_SPLIT}', '', '## Alcance', '',
    f'- Split: `{EVAL_SPLIT}`.',
    '- Modelo: ensamble weighted soft recongelado `ROBERTA_CLINICAL 512 + py RF + core RF late fusion LLM`.',
    '- Pesos: `0.65 / 0.15 / 0.20`.',
    '- No incluye texto clinico libre; los casos se identifican por `row_id` y metadatos tecnicos.',
    '', '## Resumen global', '', f'- n_eval: `{len(comparison_df)}`.',
    f'- pacientes: `{comparison_df["patient_id"].nunique()}`.',
    f'- errores del ensamble: `{int(comparison_df["error"].sum())}`.',
    f'- tasa de error: `{comparison_df["error"].mean():.6f}`.',
    '', '## Matriz de confusion del ensamble', '', cm_df.to_markdown(),
    '', '## Metricas globales por rama', '',
    branch_metrics_df[['modelo', 'n_eval', 'accuracy', 'macro_f1', 'balanced_accuracy', 'weighted_f1', 'f1_ansiedad', 'f1_depresion', 'recall_ansiedad', 'recall_depresion']].to_markdown(index=False),
    '', '## Tipos de error', '', comparison_df['tipo_error'].value_counts().rename_axis('tipo').reset_index(name='n').to_markdown(index=False),
    '', '## Casos A/B/C sugeridos', '', cases_abc_df.to_markdown(index=False),
    '', '## Pacientes con mas errores', '', patient_summary.head(10).to_markdown(index=False),
    '', '## Lectura breve', '',
    '- La diferencia entre clases y entre ramas debe interpretarse descriptivamente.',
    '- El resultado es hold-out final cuando el split es `test`; no se usa para reajustar pesos o modelos.',
]
error_report_path.write_text('\n'.join(error_report_lines) + '\n', encoding='utf-8')
display(branch_metrics_df)
display(cases_abc_df)


In [ ]:
# Reporte y manifest de corrida.
paths_out = {
    'out_dir': OUT_DIR,
    'check_alineacion': check_path,
    'predicciones_roberta_clinical': roberta_path,
    'predicciones_ensamble': pred_ens_path,
    'metricas': metrics_path,
    'metricas_por_clase': per_class_path,
    'matriz_confusion_csv': cm_path,
    'matriz_confusion_json': cm_json_path,
    'matriz_confusion_md': cm_md_path,
    'metricas_globales_por_rama': branch_metrics_path,
    'predicciones_comparadas': comparison_path,
    'errores_ensamble': errors_path,
    'resumen_por_paciente': patient_summary_path,
    'casos_abc': cases_abc_path,
    'analisis_errores': error_report_path,
    **symbolic_paths,
    **bootstrap_paths,
}
if dev_test_path is not None:
    paths_out['comparacion_dev_test'] = dev_test_path
for key, path in paths_out.items():
    paths_out[key] = rel(path)

dirty_status = git_out(['git', 'status', '--short'])
manifest = {
    'run_id': RUN_ID,
    'fecha': pd.Timestamp.now().isoformat(),
    'estado': 'TEST_EJECUTADO' if EVAL_SPLIT == 'test' else 'DRY_RUN_DEV',
    'estado_test': 'TEST_EJECUTADO_UNA_VEZ' if EVAL_SPLIT == 'test' else 'TEST_PROTEGIDO',
    'eval_split': EVAL_SPLIT,
    'commit_hash': git_out(['git', 'rev-parse', 'HEAD']),
    'dirty_status': dirty_status,
    'dataset': rel(DATA_PATH / 'dataset_denoised.csv'),
    'split_path': rel(split_path),
    'labels': LABELS,
    'modelo': 'ensamble_weighted_soft_recongelado_065_015_020',
    'pesos': {s['key']: s['peso'] for s in BRANCH_SPECS},
    'ramas': [
        {
            'key': s['key'],
            'nombre': s['nombre'],
            'tipo': s['tipo'],
            'peso': s['peso'],
            'max_length': s.get('max_length'),
            'hidden_size': s.get('hidden_size'),
            'model_path': rel(s['model_path']) if 'model_path' in s else None,
            'cols_path': rel(s['cols_path']) if 'cols_path' in s else None,
            'features_path': rel(s['features_path']) if 'features_path' in s else None,
        }
        for s in BRANCH_SPECS
    ],
    'artefactos': {k: rel(v) for k, v in ARTIFACTS.items()},
    'check_reproduccion_roberta_dev': check_repro_roberta,
    'outputs': paths_out,
    'n_eval': int(len(ens)),
    'n_pacientes': int(ens['patient_id'].nunique()),
    'metricas': metrics_row,
    'bootstrap': {
        'n': BOOTSTRAP_N,
        'seed': BOOTSTRAP_SEED,
        'agrupado_por': 'patient_id' if BOOTSTRAP_N > 0 else None,
    },
    'restricciones': [
        'sin reentrenamiento',
        'sin ajuste de pesos durante evaluacion',
        'sin modificacion de reglas clinicas',
        'sin seleccion usando test',
    ],
    'lectura_resultado_test': ({
        'macro_f1_test': metrics_row['macro_f1'],
        'balanced_accuracy_test': metrics_row['balanced_accuracy'],
        'recall_ansiedad_test': float(branch_metric_rows[0]['recall_ansiedad']),
        'recall_depresion_test': float(branch_metric_rows[0]['recall_depresion']),
        'nota': 'Resultado final hold-out; no usar para seleccionar nuevos pesos/modelos.',
    } if EVAL_SPLIT == 'test' else None),
}
manifest_path = OUT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

report_path = OUT_DIR / f'reporte_cierre_final_{EVAL_SPLIT}.md'
report_lines = [
    f'# Reporte de cierre final - split {EVAL_SPLIT}',
    '',
    f'- Run ID: `{RUN_ID}`',
    f'- Estado: `{"TEST_EJECUTADO" if EVAL_SPLIT == "test" else "DRY_RUN_DEV"}`',
    f'- Modelo: ensamble weighted soft `ROBERTA_CLINICAL 512 + py RF + core RF late fusion LLM`.',
    f'- Pesos: `0.65 / 0.15 / 0.20`.',
    f'- n_eval: `{len(ens)}`',
    f'- pacientes: `{ens["patient_id"].nunique()}`',
    '',
    '## Metricas globales',
    '',
    metrics_df.to_markdown(index=False),
    '',
    '## Metricas por clase',
    '',
    per_class_df[per_class_df['modelo'] == 'ensamble_weighted_soft'].to_markdown(index=False),
    '',
    '## Matriz de confusion',
    '',
    cm_df.to_markdown(),
    '',
    '## Metricas globales por rama',
    '',
    branch_metrics_df[['modelo', 'macro_f1', 'balanced_accuracy', 'weighted_f1', 'f1_ansiedad', 'f1_depresion']].to_markdown(index=False),
    '',
]
if BOOTSTRAP_N > 0:
    report_lines.extend([
        '## Incertidumbre por bootstrap agrupado por paciente',
        '',
        boot_summary.to_markdown(index=False),
        '',
    ])
report_lines.extend([
    '## Restricciones cumplidas',
    '',
    '- No se reentreno el modelo.',
    '- No se modificaron pesos.',
    '- No se modificaron reglas clinicas.',
    '- No se uso el split evaluado para seleccion.',
    '- La rama contextual debe haber reproducido la referencia dev antes de abrir test.',
])
report_path.write_text('\n'.join(report_lines), encoding='utf-8')

print('Manifest:', manifest_path)
print('Reporte:', report_path)
display(Markdown(report_path.read_text(encoding='utf-8')))
